# Road Network Integration & Testing — continuation
Picks up after the first real-network Dijkstra test on `Upper_Egypt_edges_weighted.parquet`.

Steps: (1) rebuild the sparse graph correctly, (2) keep the main strongly connected component, (3) snap facilities to road nodes, (4) validate against NetworkX, (5) hospital-to-hospital sanity checks.
Edit the CONFIG cell first (facility file path and column names).

In [13]:
# ---------- CONFIG ----------
from pathlib import Path
OUTPUT_DIR = Path(r"D:\Rania\Graduation Project\Recommendation system features\Dijkstra & Enhanced Dijkstra implementation\road_networks")

FACILITIES_PATH = Path(r"D:\Rania\Graduation Project\Recommendation system features\Dijkstra & Enhanced Dijkstra implementation\facilities_flagged.csv") 
LAT_COL, LON_COL, NAME_COL = "latitude_final", "longitude_final", "canonical_name" 
RELIABLE_COL = "coord_reliable"                                # flag you added earlier
SNAP_WARN_M = 1000      # flag facilities snapped farther than this from a road node
RANDOM_SEED = 42

In [14]:
import numpy as np, pandas as pd
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra, connected_components
from scipy.spatial import cKDTree

nodes = pd.read_parquet(OUTPUT_DIR / "Upper_Egypt_nodes.parquet", columns=["id", "lon", "lat"])
edges = pd.read_parquet(OUTPUT_DIR / "Upper_Egypt_edges_weighted.parquet",
                        columns=["u", "v", "length", "travel_time_s"])
print("Nodes:", f"{len(nodes):,}", "| directed edges:", f"{len(edges):,}")

Nodes: 2,300,821 | directed edges: 4,746,637


## 1. Fix parallel edges
`csr_matrix(...).sum_duplicates()` **adds** the weights of parallel edges (same `u`,`v`). Dijkstra needs the cheapest one, so collapse duplicates with `min` first.

In [15]:
dup = edges.duplicated(["u", "v"]).sum()
print("Parallel (u,v) edge records:", f"{dup:,}")

# keep the fastest parallel edge; carry its length along with it
edges = (edges.sort_values("travel_time_s")
              .drop_duplicates(["u", "v"], keep="first")
              .reset_index(drop=True))
print("Edges after collapsing parallels:", f"{len(edges):,}")

nodes = nodes.drop_duplicates("id").sort_values("id").reset_index(drop=True)
node_ids = nodes["id"].to_numpy(np.int64)
u_idx = np.searchsorted(node_ids, edges["u"].to_numpy(np.int64))
v_idx = np.searchsorted(node_ids, edges["v"].to_numpy(np.int64))
assert np.array_equal(node_ids[u_idx], edges["u"].to_numpy(np.int64))
assert np.array_equal(node_ids[v_idx], edges["v"].to_numpy(np.int64))

N = len(node_ids)
G_time = csr_matrix((edges["travel_time_s"].to_numpy(float), (u_idx, v_idx)), shape=(N, N))
G_len  = csr_matrix((edges["length"].to_numpy(float),        (u_idx, v_idx)), shape=(N, N))
print("Sparse graph:", G_time.shape, "nnz =", f"{G_time.nnz:,}")

Parallel (u,v) edge records: 0
Edges after collapsing parallels: 4,746,637
Sparse graph: (2300821, 2300821) nnz = 4,746,637


## 2. Keep the main strongly connected component
Your last test asked for a route to the last node by ID, which can sit in a disconnected fragment (clipped governorate borders). For directed routing, a node is only safely usable if it can both reach and be reached from the main network, which is what a *strongly* connected component guarantees.

In [16]:
n_comp, labels = connected_components(G_time, directed=True, connection="strong")
sizes = np.bincount(labels)
main_label = sizes.argmax()
in_main = labels == main_label

print("Strong components:", f"{n_comp:,}")
print("Main component nodes:", f"{sizes[main_label]:,}", f"({in_main.mean()*100:.2f}% of all nodes)")
print("Top 10 component sizes:", np.sort(sizes)[::-1][:10].tolist())

# where are the nodes that are NOT in the main component?
off = nodes[~in_main]
print("Nodes outside main component:", f"{len(off):,}")
print(off[["lon", "lat"]].describe().round(3).to_string())

Strong components: 5,749
Main component nodes: 2,153,108 (93.58% of all nodes)
Top 10 component sizes: [2153108, 78645, 42304, 3266, 2100, 1520, 1125, 1080, 1029, 985]
Nodes outside main component: 147,713
              lon         lat
count  147713.000  147713.000
mean       32.522      25.293
std         0.605       1.173
min        30.665      21.996
25%        32.054      24.470
50%        32.916      24.599
75%        32.956      26.240
max        33.588      29.428


In [17]:
# Re-index onto the main component only
keep_idx = np.flatnonzero(in_main)
remap = -np.ones(N, dtype=np.int64)
remap[keep_idx] = np.arange(len(keep_idx))

main_nodes = nodes.iloc[keep_idx].reset_index(drop=True)
main_ids = main_nodes["id"].to_numpy(np.int64)

mask = in_main[u_idx] & in_main[v_idx]
mu, mv = remap[u_idx[mask]], remap[v_idx[mask]]
M = len(main_nodes)

G_time_m = csr_matrix((edges["travel_time_s"].to_numpy(float)[mask], (mu, mv)), shape=(M, M))
G_len_m  = csr_matrix((edges["length"].to_numpy(float)[mask],        (mu, mv)), shape=(M, M))
print("Routing graph:", f"{M:,} nodes, {G_time_m.nnz:,} directed edges")

main_nodes.to_parquet(OUTPUT_DIR / "Upper_Egypt_main_nodes.parquet", index=False)

Routing graph: 2,153,108 nodes, 4,438,751 directed edges


## 3. Snap facilities to the nearest road node
Only rows with reliable coordinates are used (the placeholder-coordinate facilities are excluded until you have real ones).

In [18]:
R_KM = 6371.0
def to_xyz(lon, lat):
    lon, lat = np.radians(np.asarray(lon, float)), np.radians(np.asarray(lat, float))
    return np.column_stack((np.cos(lat)*np.cos(lon), np.cos(lat)*np.sin(lon), np.sin(lat)))

def haversine_km(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, (lon1, lat1, lon2, lat2))
    a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
    return 2 * R_KM * np.arcsin(np.sqrt(a))

tree = cKDTree(to_xyz(main_nodes["lon"], main_nodes["lat"]))

fac = pd.read_csv(FACILITIES_PATH)
print("Facilities in file:", len(fac))
if RELIABLE_COL in fac.columns:
    fac = fac[fac[RELIABLE_COL].astype(bool)].copy()
print("Facilities with reliable coordinates:", len(fac))

d, j = tree.query(to_xyz(fac[LON_COL], fac[LAT_COL]), k=1)
fac["node_idx"] = j
fac["node_id"]  = main_ids[j]
fac["snap_m"]   = 2 * np.arcsin(d / 2) * R_KM * 1000     # chord -> arc length
fac["snap_far"] = fac["snap_m"] > SNAP_WARN_M

print(fac["snap_m"].describe().round(1).to_string())
print("Snapped farther than", SNAP_WARN_M, "m:", int(fac["snap_far"].sum()))
fac.to_csv(OUTPUT_DIR / "facilities_snapped.csv", index=False)
fac.loc[fac["snap_far"], [NAME_COL, LAT_COL, LON_COL, "snap_m"]].sort_values("snap_m", ascending=False).head(15)

Facilities in file: 798
Facilities with reliable coordinates: 736
count      736.0
mean       219.4
std        992.4
min          0.3
25%         15.3
50%         25.9
75%         46.2
max      14052.1
Snapped farther than 1000 m: 34


,canonical_name,latitude_final,longitude_final,snap_m
664,مستشفى نصر النوبة المركزي,24.538481,33.034890,14052.076783
582,Kom Ombo General Hospital,24.484838,32.948756,6225.092949
501,مستشفي دار السلام المركزي سوهاج,26.279193,32.058060,6133.579970
512,مستشفى الدكتور خالد البطل للتوليد وأمراض النساء,26.275830,32.059023,6062.405063
504,مستشفى الدكتور خالد البطل,26.275470,32.058967,6041.577647
505,elfath medical center,26.277949,32.056806,5959.984707
502,مستشفى المصطفى التخصصي,26.275640,32.057367,5902.560620
657,مستشفى نفرتاري | Nefertari Hospital,24.479312,32.950070,5896.015443
503,الادارة الصحية بدار السلام,26.277745,32.056055,5882.821711
499,مستشفى السلام,26.277545,32.054499,5734.026928


## 4. Validation against NetworkX
Independent implementation, random node pairs from the main component. Times must agree to floating-point tolerance. If you want to test Member 1's `dijkstra_core.py` too, add it where marked.

In [19]:
import networkx as nx

G_nx = nx.from_scipy_sparse_array(G_time_m, create_using=nx.DiGraph, edge_attribute="weight")
print("NetworkX graph:", G_nx.number_of_nodes(), "nodes,", G_nx.number_of_edges(), "edges")

rng = np.random.default_rng(RANDOM_SEED)
N_PAIRS = 50
src = rng.integers(0, M, N_PAIRS)
dst = rng.integers(0, M, N_PAIRS)

rows = []
for s, t in zip(src, dst):
    if s == t:
        continue
    d_sp = dijkstra(G_time_m, directed=True, indices=int(s))[int(t)]
    d_nx = nx.dijkstra_path_length(G_nx, int(s), int(t), weight="weight")
    # ---- optional: compare Member 1's implementation here ----
    # import dijkstra_core; d_core = dijkstra_core.<function>(...)
    rows.append((int(s), int(t), d_sp, d_nx, abs(d_sp - d_nx)))

val = pd.DataFrame(rows, columns=["src_idx", "dst_idx", "scipy_s", "networkx_s", "abs_diff"])
print(val.head().to_string(index=False))
print("\nPairs tested:", len(val), "| max abs diff (s):", val["abs_diff"].max())
assert val["abs_diff"].max() < 1e-6, "scipy and NetworkX disagree"
print("PASS: scipy Dijkstra matches NetworkX on", len(val), "random pairs.")

NetworkX graph: 2153108 nodes, 4438751 edges
 src_idx  dst_idx      scipy_s   networkx_s  abs_diff
  192166  1636145 24793.663926 24793.663926       0.0
 1666410   419078  1892.596080  1892.596080       0.0
 1409363   783528 17630.875359 17630.875359       0.0
  944952  1004900  4443.346827  4443.346827       0.0
  932328  1071798 15150.498689 15150.498689       0.0

Pairs tested: 50 | max abs diff (s): 0.0
PASS: scipy Dijkstra matches NetworkX on 50 random pairs.


## 5. Hospital-to-hospital sanity checks
Road distance should be somewhat longer than straight-line distance (typically ~1.1–1.6x). Ratios below 1 are impossible; very large ones usually mean a bad snap or a missing road link.

In [20]:
def route(src_idx, dst_idx):
    """Return (node_idx_path, time_s, length_m) for the fastest route, or None if unreachable."""
    dist, pred = dijkstra(G_time_m, directed=True, indices=src_idx, return_predecessors=True)
    if not np.isfinite(dist[dst_idx]):
        return None
    path, cur = [dst_idx], dst_idx
    while cur != src_idx:
        cur = int(pred[cur]); path.append(cur)
    path.reverse()
    length_m = sum(G_len_m[a, b] for a, b in zip(path[:-1], path[1:]))
    return path, float(dist[dst_idx]), float(length_m)

sample = fac[~fac["snap_far"]].sample(n=min(12, int((~fac["snap_far"]).sum())), random_state=RANDOM_SEED).reset_index(drop=True)
src_nodes = sample["node_idx"].to_numpy()

D, P = dijkstra(G_time_m, directed=True, indices=src_nodes, return_predecessors=True)

pairs = []
for i in range(len(sample)):
    for k in range(len(sample)):
        if i == k:
            continue
        t = D[i, sample.loc[k, "node_idx"]]
        straight = haversine_km(sample.loc[i, LON_COL], sample.loc[i, LAT_COL],
                                sample.loc[k, LON_COL], sample.loc[k, LAT_COL])
        r = route(int(src_nodes[i]), int(sample.loc[k, "node_idx"])) if np.isfinite(t) else None
        road_km = r[2] / 1000 if r else np.nan
        pairs.append({
            "from": sample.loc[i, NAME_COL], "to": sample.loc[k, NAME_COL],
            "straight_km": round(straight, 1), "road_km": round(road_km, 1) if r else np.nan,
            "time_min": round(t / 60, 1) if np.isfinite(t) else np.nan,
            "ratio": round(road_km / straight, 2) if r and straight > 0 else np.nan,
        })
chk = pd.DataFrame(pairs)
print("Unreachable pairs:", int(chk["time_min"].isna().sum()), "of", len(chk))
print(chk["ratio"].describe().round(2).to_string())
print("\nRatios < 1.0 (impossible, check snapping):")
print(chk[chk["ratio"] < 1.0].to_string(index=False))
print("\nRatios > 2.0 (suspicious detours):")
print(chk[chk["ratio"] > 2.0].head(10).to_string(index=False))

Unreachable pairs: 0 of 132
count    132.00
mean       4.59
std       21.71
min        1.05
25%        1.17
50%        1.27
75%        1.65
max      228.88

Ratios < 1.0 (impossible, check snapping):
Empty DataFrame
Columns: [from, to, straight_km, road_km, time_min, ratio]
Index: []

Ratios > 2.0 (suspicious detours):
                                   from                                      to  straight_km  road_km  time_min  ratio
                      مركز الصفوة الطبي                         مستشفى المعلمين        131.1    280.6     226.9   2.14
                      مركز الصفوة الطبي                      مستشفي النيل نقادة          4.5    457.8     403.1 101.87
                      مركز الصفوة الطبي Alamia Exhibition Tools Health Algossry        115.3    298.3     242.3   2.59
                      مركز الصفوة الطبي                           مستشفى الحياة        127.6    283.9     227.8   2.22
                      مركز الصفوة الطبي                 Sohag teaching hospital     

Next: pick 3 or 4 of these pairs, look up the real driving distance on Google Maps, and note the difference in your report (travel times will differ because the speeds are still provisional defaults).

In [21]:
facilities = pd.read_csv(FACILITIES_PATH)

print("Columns:")
print(facilities.columns.tolist())

print("\nFlagged facilities:", (~facilities[RELIABLE_COL].fillna(False).astype(bool)).sum())

Columns:
['canonical_entity_id', 'canonical_name', 'governorate_assigned', 'district', 'address', 'phone_clean', 'phone_type', 'phones_other', 'phone_e164', 'latitude_final', 'longitude_final', 'facility_class', 'birth_service_tier', 'alternate_names', 'facility_category', 'birth_service_evidence', 'phone', 'phones_normalized_all', 'source_files_all', 'source_files', 'source_count', 'n_source_records_total', 'n_same_coord', 'coord_reliable']

Flagged facilities: 62


In [22]:

import pandas as pd

facilities = pd.read_csv(FACILITIES_PATH)

# Parse the reliability flag safely, including string values such as "False"
reliable = (
    facilities["coord_reliable"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["true", "1", "yes"])
)

flagged = facilities.loc[~reliable].copy()

print("Flagged facilities:", len(flagged))

display(
    flagged[
        [
            "canonical_name",
            "governorate_assigned",
            "district",
            "latitude_final",
            "longitude_final",
            "source_files_all",
            "source_count",
            "n_same_coord",
            "coord_reliable",
        ]
    ].sort_values(
        ["governorate_assigned", "n_same_coord"],
        ascending=[True, False],
        na_position="last",
    )
)


Flagged facilities: 62


,canonical_name,governorate_assigned,district,latitude_final,longitude_final,source_files_all,source_count,n_same_coord,coord_reliable
32,ديروط العام,Assiut,ديروط,27.183282,31.185384,mohp,1,9,False
35,ابنوب المركزى,Assiut,ابنوب,27.183282,31.185384,mohp,1,9,False
37,البدارى المركزى,Assiut,البدارى,27.183282,31.185384,mohp,1,9,False
38,القوصية المركزى,Assiut,القوصية,27.183282,31.185384,mohp,1,9,False
39,ساحل سليم المركزى,Assiut,ساحل سليم,27.183282,31.185384,mohp,1,9,False
...,...,...,...,...,...,...,...,...,...
56,ساقلته المركزى,Sohag,ساقلته,26.762463,32.090994,mohp,1,12,False
58,مستشفى سوهاج التعليمى,Sohag,سوهاج,26.762463,32.090994,mohp,1,12,False
404,مركز العوضي الطبي,Sohag,Sohag 2,26.559074,31.695670,scrape | registry,2,3,False
408,Regional Union For Sohag Labors Syndicates,Sohag,Sohag 2,26.559074,31.695670,scrape,1,3,False


In [23]:
%pip install folium


   ---------------------------------------- 0/3 [xyzservices]
   ---------------------------------------- 0/3 [xyzservices]
   ------------- -------------------------- 1/3 [branca]
   ------------- -------------------------- 1/3 [branca]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   ---------------------------------------- 3/3 [folium]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:

import pandas as pd
import folium
from pathlib import Path

# Load hospital locations
facilities = pd.read_csv(FACILITIES_PATH)

# Parse coordinate reliability safely
reliable = (
    facilities["coord_reliable"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["true", "1", "yes"])
)

hospitals = facilities.loc[
    reliable
    & facilities["latitude_final"].notna()
    & facilities["longitude_final"].notna()
].copy()

print("Reliable-coordinate facilities:", len(hospitals))


Reliable-coordinate facilities: 736


In [27]:

# Center the map on the hospitals' average coordinates
center_lat = hospitals["latitude_final"].mean()
center_lon = hospitals["longitude_final"].mean()

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=7,
    tiles="OpenStreetMap"
)

# Cluster markers so hundreds of hospitals remain manageable
from folium.plugins import MarkerCluster

cluster = MarkerCluster(name="Hospitals").add_to(m)

for _, row in hospitals.iterrows():
    folium.Marker(
        location=[
            row["latitude_final"],
            row["longitude_final"]
        ],
        tooltip=str(row["canonical_name"]),
        popup=(
            f"<b>{row['canonical_name']}</b><br>"
            f"Governorate: {row['governorate_assigned']}<br>"
            f"District: {row['district']}"
        )
    ).add_to(cluster)

folium.LayerControl().add_to(m)

m


In [34]:
import folium
import pandas as pd
from folium.plugins import MarkerCluster

# Ensure coordinates are numeric
hospitals["latitude_final"] = pd.to_numeric(
    hospitals["latitude_final"], errors="coerce"
)
hospitals["longitude_final"] = pd.to_numeric(
    hospitals["longitude_final"], errors="coerce"
)

hospitals = hospitals.dropna(
    subset=["latitude_final", "longitude_final"]
).copy()

# Create map with a different tile provider
center_lat = hospitals["latitude_final"].mean()
center_lon = hospitals["longitude_final"].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=8)

cluster = MarkerCluster(name="Hospitals").add_to(m)

for _, row in hospitals.iterrows():
    folium.Marker(
        location=[
            row["latitude_final"],
            row["longitude_final"]
        ],
        tooltip=str(row["canonical_name"]),
        popup=(
            f"<b>{row['canonical_name']}</b><br>"
            f"Governorate: {row['governorate_assigned']}<br>"
            f"District: {row['district']}"
        )
    ).add_to(cluster)

folium.LayerControl().add_to(m)

# Save the new map
map_path = OUTPUT_DIR / "hospital_locations_map.html"
m.save(str(map_path))

print("Facilities plotted:", len(hospitals))
print("Map saved to:", map_path.resolve())

m

Facilities plotted: 736
Map saved to: D:\Rania\Graduation Project\Recommendation system features\Dijkstra & Enhanced Dijkstra implementation\road_networks\hospital_locations_map.html


In [36]:
!pip install plotly

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.7 MB 4.2 MB/s eta 0:00:03
   ----- ---------------------------------- 1.3/9.7 MB 3.9 MB/s eta 0:00:03
   -------- ------------------------------- 2.1/9.7 MB 4.1 MB/s eta 0:00:02
   ----------- ---------------------------- 2.9/9.7 MB 3.9 MB/s eta 0:00:02
   --------------- ------------------------ 3.7/9.7 MB 4.0 MB/s eta 0:00:02
   ------------------ --------------------- 4.5/9.7 MB 3.9 MB/s eta 0:00:02
   --------------------- ------------------ 5.2/9.7 MB 3.9 MB/s eta 0:00:02
   ------------------------ --------------- 6.0/9.7 MB 3.9 MB/s eta 0:00:01
   ---------------------------- ----------- 6.8/9.7 MB 3.9 MB/s eta 0:00:01
   -------------------------------- ------- 7.9/9.7 MB 3.9 MB/s eta 0:00:01
   ----------------------------------- ---- 8.7/9.7 MB 3.9 MB/s eta 0:00:01
   -------------------------------------- - 9.4/9.7 MB 3.9 MB/s eta 0:00:01
   ----------------


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [39]:
import plotly.express as px
import pandas as pd

# Ensure coordinates are numeric
plot_data = hospitals.copy()

plot_data["latitude_final"] = pd.to_numeric(
    plot_data["latitude_final"], errors="coerce"
)
plot_data["longitude_final"] = pd.to_numeric(
    plot_data["longitude_final"], errors="coerce"
)

plot_data = plot_data.dropna(
    subset=["latitude_final", "longitude_final"]
)

# Create the interactive hospital map
fig = px.scatter_map(
    plot_data,
    lat="latitude_final",
    lon="longitude_final",
    hover_name="canonical_name",
    hover_data=["governorate_assigned", "district"],
    zoom=7,
    height=650,
    map_style="open-street-map",
)

fig.update_layout(
    map_center={
        "lat": plot_data["latitude_final"].mean(),
        "lon": plot_data["longitude_final"].mean(),
    },
    margin={"r": 0, "t": 40, "l": 0, "b": 0},
    title="Hospital Locations — Upper Egypt",
)

map_path = OUTPUT_DIR / "hospital_locations_plotly.html"
fig.write_html(str(map_path), include_plotlyjs=True)

print("Hospitals plotted:", len(plot_data))
print("Saved to:", map_path.resolve())

fig.show()

Hospitals plotted: 736
Saved to: D:\Rania\Graduation Project\Recommendation system features\Dijkstra & Enhanced Dijkstra implementation\road_networks\hospital_locations_plotly.html
